# How Learning Can Guide Evolution Recreation

The following notebook is an attempt at recreating Hinton and Nowlan's paper that proves the relevance of the Baldwin effect with a simple example, using alleles with 20 genes. Said genes have three possible states 1, for the existence of said trait, 0, for the lack thereof, or ? for the possibility of being open to learning or guessing during the organism's lifetime. 

If the organism has a ? gene it should receive multiple lifetime trials to guess/learn the correct setting for the open slots containing the sign. If it figures out the correct configuration, it gets rewarded with high fitness. What it learns is not written back into the genes. 

The idea here is that the learning gives a survival advantage to organisms that have **some** of the correct genes preset, and attribute the remaining work to learning. 
Instead of a flat landscape where only the perfect genotype survives, learning should introduce the possibility for organisms with a good genettic start to learn the rest quickly and survive. 
Over generations natiral selection should favor genotypes with the correct 1s because it reduces the cost of learning from scratch. Eventually the hardcoded ? becomes 1. 

``The "training a few epochs" part isn't the mechanism that resolves ? — it's just the unavoidable cost of finding out whether this particular random guess was any good, since (unlike the original paper's instant bit-comparison) you can't know if a hyperparameter combo is good without actually running it. If it doesn't clear the threshold, throw that whole trained network away and try again with a fresh random draw of the ? bits — don't continue training the same network, don't nudge it toward the previous guess. Each trial is a fully independent roll of the dice, exactly like the original.``

One organism's lifetime, end to end

Genotype fixes some bits, leaves others ?.

For trial = 1 to max_trials

Randomly resample all ? bits.

Decode the complete bit string → architecture + hyperparameters.

Build and briefly train that NN from scratch.

Evaluate on held-out validation data. Clears threshold? → record trial number, stop lifetime, done.

Doesn't clear it? → discard, next trial.

If no trial succeeds within the budget → treat as failure, low fitness.

Fitness = increasing function of "cleared the threshold" and decreasing function of "how many trials it took" (or a fixed low value if it never succeeded).

## OPTIONS FOR HYPERPARAMETERS

| Hyperparameter |	Bits |	Menu (decoded by bit value) |
| -------- | ------- | ------- |
| n_hidden_layers	| 2	| 1, 2, 3, 4 |
| units_per_layer	| 3	| 8, 16, 32, 64, 128, 256, 512, 1024 |
| activation	| 1	| relu, tanh |
| learning_rate	| 2	| 0.1, 0.01, 0.001, 0.0001 |
| batch_size	| 2	| 16, 32, 64, 128 |
| dropout_rate	| 2	| 0.0, 0.1, 0.2, 0.3 |

## Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pandas.plotting import scatter_matrix

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import MaxAbsScaler
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import TargetEncoder

from sklearn.decomposition import TruncatedSVD
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_regression

from sklearn.compose import TransformedTargetRegressor

from sklearn.base import BaseEstimator, TransformerMixin

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_validate
from sklearn.model_selection import cross_val_score

from sklearn.neural_network import MLPRegressor

from sklearn.metrics import mean_absolute_error

In [2]:
rng = np.random.RandomState(2)

## Defining genes

Below we specify the possible genes that can will be used to define the NN. Each hyperparameter has a variable amounts of genes that represent it, according to the possible options for it (eg hidden layer only requires 2 bits whilst units per layer requires 3)

A genome is a sequence of bits where each bit has possible values

1 -> fixed bit

0 -> fixed bit

None -> ? gene, unresolved and guessed on each trial.


The sequence of bits will be used to decode and encode the options for the hyperparameter.

In [3]:
GENE_SPEC = [
    ("n_hidden_layers", 2, [1, 2, 3, 4]),
    ("units_per_layer", 3, [8, 16, 32, 64, 128, 256, 512, 1024]),
    ("activation", 1, ["relu", "tanh"]),
    ("learning_rate", 2, [0.0001, 0.001, 0.01, 0.1]),
    ("batch_size", 2, [16, 32, 64,128]),
    ("alpha", 2, [0.0001, 0.001, 0.01, 0.1]),
    ("solver", 1, ["adam", "sgd"])
]

GENOME_LENGTH = sum(bits for _, bits, _ in GENE_SPEC) #13 bits

In [4]:
def randomize_genome(p_learn=0.3):
    """Randomly generate a genome with equal probabilities of 0 and 1, 
    and some bits missing (None - ?) based on the probability of learning"""
    rng = np.random.default_rng()
    genome = []
    for _ in range(GENOME_LENGTH):
        if rng.random() <= p_learn:
            genome.append(None)
        else:
            genome.append(rng.integers(0, 2))
    return genome

In [5]:
def resolve(genome):
    """One lifetime worth of guessing the missing bits in a genome. 
    Fixed bits are preserved, while missing bits are randomly guessed.
    Returns a conrete bit list without any Nones which will then be decoded""" 
    rng = np.random.default_rng()
    return [int(rng.integers(0,2)) if bit is None else bit for bit in genome]

In [6]:
def decode(bits):
    """Return a conrete hyperparameter dictionary from a randomized concrete bit list (all 0/1)"""
    hp = {}
    pos = 0 
    for name, n_bits, menu in GENE_SPEC:
        segment = bits[pos:pos+n_bits]
        pos += n_bits
        index = int("".join(str(b) for b in segment), 2) # binary to int
        hp[name] = menu[index]
    return hp 

In [7]:
def visualize_genome(genome):
    """Visualize a genome as a string of 0/1/N"""
    return " ".join("?" if b is None else str(b) for b in genome)

## Importing and Handling Data

FROM LAB D:
## The business problem

A diamond merchant finds that her employees are quite inconsistent when pricing the diamonds they buy and sell. Perhaps if there were a pricing tool that they could consult - to get a guideline price - then there would be greater consistency across her employees. She asks you to build the tool. Most important to her is correct pricing - but if the tool can explain its prices, so much the better.

## Performance measures

You realise that this is a regression problem and that mean absolute error (MAE) is the main metric to use in the evaluation. It makes sense to compare against a baseline - a dummy regressor that predicts the mean price of the training examples.

## Dataset

The merchant provides you with a CSV file (`diamonds.csv`), each row of which describes a diamond and its price. (In truth, the dataset comes from [Kaggle](https://www.kaggle.com/datasets/ronil8/diamond-price-prediction-dataset).)

Each row in the CSV file contains 10 pieces of data, which she explains as follows:

- Carat: The carat value of a diamond is a measure of its weight. One carat is 200 milligranms.
- Cut: The cut type of a diamond (Fair, Good, Very Good, Premium, Ideal) is a measure of quality: how it interacts with light.
- Colour: The colour grade of a diamond, in this case from J -> worst to D -> best.
- Clarity: The clarity of a diamond in terms of transparency, in this case from I1 -> worst through SI2, SI1, VS2, VS1, VVS2, VVS1, to IF -> best.
- Depth: The depth of a diamond is calculated as (Height * 200) / (Width + Length). It's a percentage and, apparently, 59-62% is ideal.
- Table: The table of a diamond refers to the flat facet that you can see when you look at the diamond from above. The value recorded here is the width of this facet times 100 divided by the Width of the diamond. Apparently, 53-58% is ideal.
- Price: This is the target value. It is given in US dollars.
- Width: Millimetres.
- Length: Millimetres.
- Height: Millimetres.

In [8]:

import os
base_dir = ".."
dataset_dir = os.path.join(base_dir, "datasets")
df = pd.read_csv(os.path.join(dataset_dir, "diamonds.csv"))

In [9]:
df.head()
df.shape
df.info()
df.describe(include="all")

<class 'pandas.DataFrame'>
RangeIndex: 53940 entries, 0 to 53939
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Carat    53940 non-null  float64
 1   Cut      53940 non-null  str    
 2   Colour   53940 non-null  str    
 3   Clarity  53940 non-null  str    
 4   Depth    53940 non-null  float64
 5   Table    53940 non-null  float64
 6   Price    53940 non-null  int64  
 7   Length   53940 non-null  float64
 8   Width    53940 non-null  float64
 9   Height   53940 non-null  float64
dtypes: float64(6), int64(1), str(3)
memory usage: 4.1 MB


,Carat,Cut,Colour,Clarity,Depth,Table,Price,Length,Width,Height
count,53940.000000,53940,53940,53940,53940.000000,53940.000000,53940.000000,53940.000000,53940.000000,53940.000000
unique,NaN,5,7,8,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,Ideal,G,SI1,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,21551,11292,13065,NaN,NaN,NaN,NaN,NaN,NaN
mean,0.797940,NaN,NaN,NaN,61.749405,57.457184,3932.799722,5.731157,5.734526,3.538734
std,0.474011,NaN,NaN,NaN,1.432621,2.234491,3989.439738,1.121761,1.142135,0.705699
min,0.200000,NaN,NaN,NaN,43.000000,43.000000,326.000000,0.000000,0.000000,0.000000
25%,0.400000,NaN,NaN,NaN,61.000000,56.000000,950.000000,4.710000,4.720000,2.910000
50%,0.700000,NaN,NaN,NaN,61.800000,57.000000,2401.000000,5.700000,5.710000,3.530000
75%,1.040000,NaN,NaN,NaN,62.500000,59.000000,5324.250000,6.540000,6.540000,4.040000


In [10]:
bad_rows = (
    (df['Length'] == 0) |
    (df['Width'] == 0) |
    (df['Height'] == 0) |
    (df['Width'] > 20) |
    (df['Height'] > 20)
)

df= df[~bad_rows].reset_index(drop=True)

## Handling Model

In [11]:
features = ["Carat", "Cut", "Colour", "Clarity", "Depth", "Table", "Length", "Width", "Height"]
train_val, test = train_test_split(df, test_size=0.2, random_state=rng)
train, val = train_test_split(train_val, test_size=0.2, random_state=rng)

categorical = ["Cut", "Colour", "Clarity"]
numeric = ["Carat", "Depth", "Table", "Length", "Width", "Height"]

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
    ("num", StandardScaler(), numeric),
]).fit(train[features])

X_train = preprocessor.transform(train[features])
X_val   = preprocessor.transform(val[features])   
X_test = preprocessor.transform(test[features])
y_train = train["Price"]
y_val   = val["Price"] 
y_test = test["Price"]

In [12]:
y_scaler = StandardScaler().fit(y_train.values.reshape(-1, 1))

y_train_raw = y_train
y_train = y_scaler.transform(y_train.values.reshape(-1, 1)).ravel()

In [13]:
def train_model(X_train, y_train, X_val, y_val, hp, epochs=15, seed=rng):
    """Train a model with the given hyperparameters and return the trained model"""
    model = MLPRegressor(
        hidden_layer_sizes=tuple([hp["units_per_layer"]] * hp["n_hidden_layers"]),
            activation=hp["activation"],
            learning_rate_init=hp["learning_rate"],
            batch_size=hp["batch_size"],
            alpha=hp["alpha"],
            max_iter=epochs,
            solver=hp["solver"],
            early_stopping=False,
            random_state=seed,
            verbose = False,
        )
    with np.errstate(over="raise", invalid="raise"):
        model.fit(X_train, y_train)
    # model.fit(X_train, y_train)
    predictions_scaled = model.predict(X_val)
    predictions = y_scaler.inverse_transform(predictions_scaled.reshape(-1,1)).ravel()
    mae = mean_absolute_error(y_val, predictions)
    return model, mae

In [14]:
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FutureTimeoutError

def train_model_with_timeout(X_train, y_train, X_val, y_val, hp, epochs=15, seed=None, timeout=20):
    with ThreadPoolExecutor(max_workers=1) as executor:
        future = executor.submit(train_model, X_train, y_train, X_val, y_val, hp, epochs, seed)
        try:
            return future.result(timeout=timeout)
        except FutureTimeoutError:
            return None, float("inf")

In [15]:
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=RuntimeWarning, module="sklearn")
warnings.filterwarnings("ignore", category=ConvergenceWarning, module="sklearn")

In [16]:
from sklearn.dummy import DummyRegressor

baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)
baseline_mae = mean_absolute_error(y_val, baseline.predict(X_val))
print("baseline MAE:", baseline_mae)

for _ in range(25):
    genome = randomize_genome(p_learn=0.3)
    print(visualize_genome(genome))
    hp = decode(resolve(genome))
    try:
        _, mae = train_model_with_timeout(X_train, y_train, X_val, y_val, hp)
    except Exception:
        mae = float("inf")
    #catch nan or inf thatt don't raise exception
    if not np.isfinite(mae):
        mae = float("inf")

    print(hp, "->", mae)

baseline MAE: 3982.2307870638692
? 0 1 0 0 0 0 ? 1 ? ? 0 0
{'n_hidden_layers': 3, 'units_per_layer': 128, 'activation': 'relu', 'learning_rate': 0.001, 'batch_size': 64, 'alpha': 0.01, 'solver': 'adam'} -> 304.5271479878308
0 1 1 1 ? 0 ? 1 1 1 1 1 0
{'n_hidden_layers': 2, 'units_per_layer': 1024, 'activation': 'relu', 'learning_rate': 0.001, 'batch_size': 128, 'alpha': 0.1, 'solver': 'adam'} -> inf
? 1 0 1 0 0 ? ? 0 ? 1 0 ?
{'n_hidden_layers': 2, 'units_per_layer': 32, 'activation': 'relu', 'learning_rate': 0.1, 'batch_size': 32, 'alpha': 0.01, 'solver': 'sgd'} -> 506.8261535155616
? 0 0 1 ? 0 ? ? 0 1 ? ? ?
{'n_hidden_layers': 1, 'units_per_layer': 32, 'activation': 'relu', 'learning_rate': 0.001, 'batch_size': 32, 'alpha': 0.01, 'solver': 'sgd'} -> 437.0234421755974
? 1 ? 0 0 1 0 ? ? 0 1 1 ?
{'n_hidden_layers': 2, 'units_per_layer': 128, 'activation': 'tanh', 'learning_rate': 0.0001, 'batch_size': 16, 'alpha': 0.1, 'solver': 'sgd'} -> 654.9059469412085
0 1 1 1 1 ? 1 ? 1 0 1 0 ?
{'n_hi

Above I wanted to view what MAE the dummy regressor would return, and compare that to the result of 10 trained NNs with randomized genomes, to try to establish a baseline that a specific genome (configuration) will have to surpass to be deemed worthy and have fitness. Fitness will be measured by the genome reaching that threshold and the number of trials it took to do so (Less trials = fitter) 

**Difference with Hinton's paper --> they were looking for a specific configuration whereas this is just a threshold under which an NN is deemed worthy.** 

However I came upon a problem with the following random configuration:

```
{'n_hidden_layers': 2, 'units_per_layer': 64, 'activation': 'tanh', 'learning_rate': 0.01, 'batch_size': 32, 'alpha': 0.0001, 'solver': 'adam'} -> 684.6565372170666

c:\Users\Carlos\FYP\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:356: RuntimeWarning: overflow encountered in dot
  values += np.dot(s, s)
c:\Users\Carlos\FYP\.venv\Lib\site-packages\numpy\_core\_methods.py:132: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)
c:\Users\Carlos\FYP\.venv\Lib\site-packages\sklearn\neural_network\_base.py:188: RuntimeWarning: overflow encountered in square
  0.5 * np.average((y_true - y_pred) ** 2, weights=sample_weight, axis=0).mean()
c:\Users\Carlos\FYP\.venv\Lib\site-packages\sklearn\utils\extmath.py:229: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
c:\Users\Carlos\FYP\.venv\Lib\site-packages\sklearn\neural_network\_base.py:139: RuntimeWarning: invalid value encountered in multiply
  delta *= 1 - Z**2
```

This was likely a result of the tanh activation and high learning rate. To fix the problem I considered whether to alternate the configuration and remove some hyperparameter options, but the purpose of the experiment is to let selection discover which values are wortwhile and thus decided against it. Removing would be doing the genetic algorithm's job.

This was fixed by scaling the target when training the model and adding guards in testing to prevent crashes.
This is valuable to avoid when handling the real population which will be costly to run

RANDOM GENE RESULTS (10 trials): 315, 329, 341, 344, 389, 408, 516, 591, 756, 757 

All of these beat the 3982 dummy baseline. 

baseline MAE: 3982.2307870638692
? 0 1 0 0 0 0 ? 1 ? ? 0 0
{'n_hidden_layers': 3, 'units_per_layer': 128, 'activation': 'relu', 'learning_rate': 0.001, 'batch_size': 64, 'alpha': 0.01, 'solver': 'adam'} -> 304.5271479878308
0 1 1 1 ? 0 ? 1 1 1 1 1 0
{'n_hidden_layers': 2, 'units_per_layer': 1024, 'activation': 'relu', 'learning_rate': 0.001, 'batch_size': 128, 'alpha': 0.1, 'solver': 'adam'} -> inf
? 1 0 1 0 0 ? ? 0 ? 1 0 ?
{'n_hidden_layers': 2, 'units_per_layer': 32, 'activation': 'relu', 'learning_rate': 0.1, 'batch_size': 32, 'alpha': 0.01, 'solver': 'sgd'} -> 506.8261535155616
? 0 0 1 ? 0 ? ? 0 1 ? ? ?
{'n_hidden_layers': 1, 'units_per_layer': 32, 'activation': 'relu', 'learning_rate': 0.001, 'batch_size': 32, 'alpha': 0.01, 'solver': 'sgd'} -> 437.0234421755974
? 1 ? 0 0 1 0 ? ? 0 1 1 ?
{'n_hidden_layers': 2, 'units_per_layer': 128, 'activation': 'tanh', 'learning_rate': 0.0001, 'batch_size': 16, 'alpha': 0.1, 'solver': 'sgd'} -> 654.9059469412085
0 1 1 1 1 ? 1 ? 1 0 1 0 ?
{'n_hidden_layers': 2, 'units_per_layer': 1024, 'activation': 'relu', 'learning_rate': 0.1, 'batch_size': 64, 'alpha': 0.01, 'solver': 'adam'} -> inf
? ? ? ? ? 1 0 ? 1 ? 0 ? 0
{'n_hidden_layers': 3, 'units_per_layer': 16, 'activation': 'tanh', 'learning_rate': 0.0001, 'batch_size': 128, 'alpha': 0.0001, 'solver': 'adam'} -> 464.1044655214805
1 1 ? ? 0 0 0 1 0 1 ? ? 1
{'n_hidden_layers': 4, 'units_per_layer': 128, 'activation': 'relu', 'learning_rate': 0.001, 'batch_size': 32, 'alpha': 0.0001, 'solver': 'sgd'} -> 351.4151787204334
1 1 0 0 0 0 1 1 0 0 0 0 0
{'n_hidden_layers': 4, 'units_per_layer': 8, 'activation': 'relu', 'learning_rate': 0.1, 'batch_size': 16, 'alpha': 0.0001, 'solver': 'adam'} -> 608.3530183657991
? 1 0 ? 1 1 ? 1 ? 1 ? 0 ?
{'n_hidden_layers': 2, 'units_per_layer': 16, 'activation': 'tanh', 'learning_rate': 0.1, 'batch_size': 32, 'alpha': 0.01, 'solver': 'adam'} -> 760.3297120177122
? 1 0 ? 0 ? ? 0 0 0 1 1 0
{'n_hidden_layers': 4, 'units_per_layer': 8, 'activation': 'relu', 'learning_rate': 0.0001, 'batch_size': 16, 'alpha': 0.1, 'solver': 'adam'} -> 389.288299350865
1 1 ? ? 1 0 1 0 0 ? ? 1 0
{'n_hidden_layers': 4, 'units_per_layer': 64, 'activation': 'relu', 'learning_rate': 0.01, 'batch_size': 16, 'alpha': 0.001, 'solver': 'adam'} -> inf
1 0 1 1 ? 1 0 0 ? 0 ? ? 1
{'n_hidden_layers': 3, 'units_per_layer': 1024, 'activation': 'tanh', 'learning_rate': 0.0001, 'batch_size': 64, 'alpha': 0.0001, 'solver': 'sgd'} -> inf
1 ? 0 0 1 1 0 1 ? ? 1 1 1
{'n_hidden_layers': 4, 'units_per_layer': 16, 'activation': 'tanh', 'learning_rate': 0.001, 'batch_size': 128, 'alpha': 0.1, 'solver': 'sgd'} -> 505.9944277080863
1 ? 0 ? 1 0 0 1 1 0 1 0 1
{'n_hidden_layers': 3, 'units_per_layer': 64, 'activation': 'relu', 'learning_rate': 0.001, 'batch_size': 64, 'alpha': 0.01, 'solver': 'sgd'} -> 411.3142793933406
0 1 1 ? ? 1 1 0 ? 0 1 1 ?
{'n_hidden_layers': 2, 'units_per_layer': 128, 'activation': 'tanh', 'learning_rate': 0.01, 'batch_size': 16, 'alpha': 0.1, 'solver': 'adam'} -> 785.2312143852854
1 1 ? ? 1 0 ? 1 ? 1 1 ? 1
{'n_hidden_layers': 4, 'units_per_layer': 1024, 'activation': 'relu', 'learning_rate': 0.1, 'batch_size': 32, 'alpha': 0.01, 'solver': 'sgd'} -> inf
? 0 ? 1 1 1 1 1 1 1 1 ? 1
{'n_hidden_layers': 1, 'units_per_layer': 1024, 'activation': 'tanh', 'learning_rate': 0.1, 'batch_size': 128, 'alpha': 0.1, 'solver': 'sgd'} -> inf
? 1 0 ? ? 1 1 0 ? 0 0 ? 0
{'n_hidden_layers': 4, 'units_per_layer': 8, 'activation': 'tanh', 'learning_rate': 0.01, 'batch_size': 16, 'alpha': 0.001, 'solver': 'adam'} -> 436.15835893523297
? 0 1 0 1 0 ? 0 0 ? ? 0 ?
{'n_hidden_layers': 1, 'units_per_layer': 256, 'activation': 'relu', 'learning_rate': 0.01, 'batch_size': 32, 'alpha': 0.01, 'solver': 'adam'} -> 459.69177969862477
? 0 0 1 0 1 0 0 ? 1 ? 1 0
{'n_hidden_layers': 1, 'units_per_layer': 32, 'activation': 'tanh', 'learning_rate': 0.0001, 'batch_size': 32, 'alpha': 0.001, 'solver': 'adam'} -> 432.66176822826094
0 0 1 1 1 ? ? 0 ? 0 0 ? 0
{'n_hidden_layers': 1, 'units_per_layer': 1024, 'activation': 'relu', 'learning_rate': 0.0001, 'batch_size': 64, 'alpha': 0.001, 'solver': 'adam'} -> 328.5789016485196
? ? 0 1 1 ? 0 0 1 ? 1 1 0
{'n_hidden_layers': 2, 'units_per_layer': 64, 'activation': 'relu', 'learning_rate': 0.0001, 'batch_size': 128, 'alpha': 0.1, 'solver': 'adam'} -> 385.53353359964814
1 0 1 0 ? 1 0 1 0 0 ? ? 1
{'n_hidden_layers': 3, 'units_per_layer': 256, 'activation': 'tanh', 'learning_rate': 0.001, 'batch_size': 16, 'alpha': 0.1, 'solver': 'sgd'} -> inf
0 1 1 ? ? 1 ? 0 0 1 ? 0 ?
{'n_hidden_layers': 2, 'units_per_layer': 512, 'activation': 'tanh', 'learning_rate': 0.01, 'batch_size': 32, 'alpha': 0.0001, 'solver': 'adam'} -> inf
